In [0]:
orders_df = spark.read.format("delta").load("/Volumes/pei/silver/orders/")

In [0]:
customers_df = spark.read.format("delta").load("/Volumes/pei/silver/customers/")

In [0]:
products_df = spark.read.format("delta").load("/Volumes/pei/silver/products/")

Enriched table with order, customer and products details

In [0]:
from pyspark.sql.functions import round, col, broadcast

fact_orders = (
    orders_df
    .join(customers_df, "customer_id", "inner")
    .join(broadcast(products_df), "product_id", "inner")
    .withColumn("profit", round(col("profit"), 2))
    .drop("ingestion_timestamp", "product_price"))

fact_orders.write.format("delta").mode("overwrite").save("/Volumes/pei/gold/fact_orders/")

Aggregated profitibilty by year, category, subcategory, customer_Id

In [0]:
from pyspark.sql.functions import year, sum

agg_profit = (
    fact_orders
    .withColumn("year", year(col("order_date")))
    .groupBy(
        "year",
        "category",
        "subcategory",
        "customer_Id"
    )
    .agg(sum("profit").alias("total_profit"))
)
display(agg_profit)

In [0]:
fact_orders.createOrReplaceTempView("fact_orders")

Profit per year

In [0]:
%sql
SELECT year(order_date) AS year, round(SUM(profit),2) AS total_profit
FROM fact_orders
GROUP BY year

Profit per year & per category

In [0]:
%sql
SELECT year(order_date) as year, category, ROUND(SUM(profit),2) as profit
FROM fact_orders
GROUP BY year(order_date), category

Profit per customer

In [0]:
%sql
SELECT customer_id, ROUND(SUM(profit),2) AS profit_per_customer
FROM fact_orders
GROUP BY customer_id

Profit per year, per customer

In [0]:
%sql
SELECT  year(order_date) as year, customer_id, round(SUM(profit),2) as profit
FROM fact_orders
GROUP BY year(order_date),  customer_id